<a href="https://colab.research.google.com/github/nimishakavisrinithya/24P31A05B3_DriveReady_OPC_2026/blob/main/Day3/agenticAI_task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

from langchain_groq import ChatGroq

In [18]:
!pip install -q langchain langchain-groq

In [19]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes—plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!
Your name is Ravi.


In [21]:
history

[('human', "Hi, I'm Ravi. Do you make eggless cakes?"),
 ('ai',
  'Hi Ravi! 😊 Yes, we offer egg‑less versions of all our cakes—plum, chocolate, and cupcakes. Let me know which one you’d like, and I’ll help you with the details!'),
 ('human', "What's my name?"),
 ('ai', 'Your name is Ravi.')]

In [22]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

In [25]:
import csv

rows = [
    {"question": "Do you make eggless chocolate cake?", "must_include": "egg"},
    {"question": "How much are 2 cupcakes?", "must_include": "160"},
    {"question": "Can you deliver at 5 PM today?", "must_include": "check with the owner"},
]

with open("/content/test.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["question", "must_include"])
    writer.writeheader()
    writer.writerows(rows)

In [26]:
run_tests()

PASS | Do you make eggless chocolate cake? → Yes! We offer an egg‑less chocolate cake (Rs 650). Let me know if you’
PASS | How much are 2 cupcakes? → Two cupcakes will cost Rs 160. Let me know if you'd like to add anythi
PASS | Can you deliver at 5 PM today? → I'll check with the owner and get back to you soon!

3/3 passed (100%)


In [29]:
from langchain.agents import create_agent

agent_order = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result_order = agent_order.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result_order["messages"][-1].content)

Yes! All our cakes, including plum, chocolate, and cupcakes, are available in egg‑less versions. Let us know which one you’d like!
